# 01 - Exploratory data analysis (main corpus, 50,000 RecipeNLG recipes)

Reads `data/processed/main/` (built by `python -m src.data_collection`). Every number below is computed
in this notebook from the processed files; figures are also written to `reports/figures/`.
Run it from a clean kernel: `Restart & Run All`.

In [ ]:
import collections
import json
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():  # started from notebooks/
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed" / "main"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


corpus = read_jsonl(PROCESSED / "corpus.jsonl")
manifest = json.loads((PROCESSED / "dataset_manifest.json").read_text(encoding="utf-8"))
frame = pd.DataFrame(
    {
        "recipe_id": [r["recipe_id"] for r in corpus],
        "group_id": [r["group_id"] for r in corpus],
        "source": [r["source"] for r in corpus],
        "n_ingredients": [len(r["ingredients_normalized"]) for r in corpus],
        "n_steps": [len(r["directions"]) for r in corpus],
        "direction_chars": [sum(len(s) for s in r["directions"]) for r in corpus],
    }
)
print(len(frame), "recipes;", manifest["raw"]["rows"], "rows in the raw CSV;", manifest["counts"])
print("dropped while selecting:", manifest["selection"]["drop_reasons"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(frame["n_ingredients"], bins=range(1, frame["n_ingredients"].max() + 2), color="#2F5496", align="left")
ax.set_xlabel("normalised ingredients per recipe")
ax.set_ylabel("recipes")
ax.set_title("Ingredients per recipe")
fig.tight_layout()
fig.savefig(FIGURES / "eda_1_ingredients_per_recipe.png")
plt.show()
q = frame["n_ingredients"].quantile([0.5, 0.95])
print(f"mean {frame['n_ingredients'].mean():.2f}, median {q[0.5]:.0f}, p95 {q[0.95]:.0f}, max {frame['n_ingredients'].max()}")

**Reading:** recipes are short: the median is 8 ingredients and 95% have 15 or fewer, with a thin tail up to 37
(the pipeline drops recipes with fewer than 2 or more than 40). A 2-6 ingredient query therefore covers a large
share of a typical recipe, which is why hiding at least one ingredient when generating queries matters.

In [ ]:
doc_freq = collections.Counter(i for r in corpus for i in set(r["ingredients_normalized"]))
top = doc_freq.most_common(25)
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.barh([name for name, _ in top][::-1], [100 * count / len(corpus) for _, count in top][::-1], color="#2F5496")
ax.set_xlabel("% of recipes containing the ingredient")
ax.set_title("25 most common ingredients")
fig.tight_layout()
fig.savefig(FIGURES / "eda_2_top_ingredients.png")
plt.show()
common = sum(1 for count in doc_freq.values() if count > 0.02 * len(corpus))
print(f"vocabulary {len(doc_freq)}; names in >2% of recipes: {common}; top-1: {top[0][0]} ({100*top[0][1]/len(corpus):.1f}%)")

**Reading:** salt appears in 43% of recipes and sugar, egg, butter, onion and flour in 20-27% each, so a few pantry
staples dominate. Only 79 names occur in more than 2% of recipes; this is the threshold used to call an ingredient
"common" when generating queries (every query needs at least one non-common ingredient).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(frame["n_steps"].clip(upper=30), bins=range(1, 32), color="#2F5496", align="left")
axes[0].set_xlabel("direction steps (clipped at 30)")
axes[0].set_ylabel("recipes")
axes[0].set_title("Steps per recipe")
axes[1].hist(frame["direction_chars"].clip(upper=3000), bins=60, color="#2F5496")
axes[1].set_xlabel("characters of directions (clipped at 3,000)")
axes[1].set_title("Directions length")
fig.tight_layout()
fig.savefig(FIGURES / "eda_3_directions.png")
plt.show()
print(frame[["n_steps", "direction_chars"]].describe(percentiles=[0.5, 0.95]).round(1).T)

**Reading:** directions are short (median 5 steps, about 350 characters) with a long tail of very detailed recipes
(up to 106 steps). Directions are only displayed in the app; the encoder sees ingredients (and optionally the title),
so this tail does not affect retrieval or truncation.

In [ ]:
counts = frame["source"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].bar(counts.index, counts.values, color=["#2F5496", "#8FAADC"][: len(counts)])
axes[0].set_ylabel("recipes")
axes[0].set_title("Source")
frame.boxplot(column="n_ingredients", by="source", ax=axes[1], grid=False)
axes[1].set_title("Ingredients per recipe by source")
axes[1].set_xlabel("")
plt.suptitle("")
fig.tight_layout()
fig.savefig(FIGURES / "eda_4_sources.png")
plt.show()
print(counts.to_dict(), frame.groupby("source")["n_ingredients"].mean().round(2).to_dict())

**Reading:** 74% of the sample comes from `Gathered` (web/cookbook text) and 26% from `Recipes1M`. Recipes1M recipes
have slightly more ingredients (8.7 vs 8.1). The raw file is sorted by source, which is why the sample is drawn from
the whole file instead of its first rows.

In [ ]:
sizes = frame["group_id"].value_counts()
size_hist = sizes.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.loglog(size_hist.index, size_hist.values, "o", color="#2F5496")
ax.set_xlabel("recipes per group (identical ingredient set or title)")
ax.set_ylabel("number of groups")
ax.set_title("Duplicate-group sizes (log-log)")
fig.tight_layout()
fig.savefig(FIGURES / "eda_5_group_sizes.png")
plt.show()
print(f"{len(sizes)} groups; singletons {(sizes == 1).sum()} ({100*(sizes == 1).mean():.1f}%); largest group {sizes.max()}")

**Reading:** 93% of groups hold a single recipe, but a heavy tail has groups of over 100 near-identical recipes
(same title or same ingredient set; the largest has 141). Splitting by group keeps all of them on one side; a random
recipe-level split would leak these near-duplicates between train and test.

In [ ]:
ranked = np.array(sorted(doc_freq.values(), reverse=True))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.loglog(np.arange(1, len(ranked) + 1), ranked, color="#2F5496")
ax.set_xlabel("ingredient rank")
ax.set_ylabel("recipes containing it")
ax.set_title("Ingredient frequency (rank-frequency)")
fig.tight_layout()
fig.savefig(FIGURES / "eda_6_rank_frequency.png")
plt.show()
share_top100 = sum(c for c in sorted(doc_freq.values(), reverse=True)[:100]) / sum(doc_freq.values())
singletons = int((ranked == 1).sum())
print(f"top-100 names cover {100*share_top100:.1f}% of ingredient mentions; singletons {singletons} ({100*singletons/len(ranked):.1f}% of vocabulary)")

**Reading:** the vocabulary is Zipf-like: 100 names cover 64% of all mentions while 60% of the 14,450 distinct names
appear in a single recipe. Rare names carry the signal for retrieval (high IDF) but many singletons are noisy
NER fragments, which the audit below documents.

## NER and normalisation checks

The full manual review of 100 recipes (seed 42) with recipe ids is in `reports/data_audit.md`. Two reproducible checks:

In [ ]:
import random

rng = random.Random(42)
sample = rng.sample(corpus, 100)
rows = [
    {"recipe_id": r["recipe_id"], "ner": ", ".join(r["ner_raw"]), "normalised": ", ".join(r["ingredients_normalized"])}
    for r in sample
]
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(rows).head(10)

In [ ]:
# The normaliser is not a fixed point for some names: normalising an already-normalised name changes it.
import sys

sys.path.insert(0, str(ROOT))
from src.normalize import normalize_ingredients  # noqa: E402

changed = {n: normalize_ingredients([n]) for n in doc_freq if normalize_ingredients([n]) != [n]}
affected = sum(1 for r in corpus if any(i in changed for i in r["ingredients_normalized"]))
print(f"{len(changed)} of {len(doc_freq)} distinct names change on a second pass; {affected} recipes ({100*affected/len(corpus):.2f}%) contain one")
print(list(changed.items())[:8])

fewer = sum(1 for r in corpus if len(r["ner_raw"]) < len(r["ingredients_raw"]))
print(f"NER has fewer entries than raw ingredient lines in {fewer} recipes ({100*fewer/len(corpus):.1f}%)")

**Reading:** about 1.4% of distinct names (0.8% of recipes) are changed by a second normalisation pass, and in one
recipe out of five the NER column lists fewer items than the raw ingredient lines. Retrieval stays consistent
(documents and queries receive the same number of passes) but these are real normalisation errors; see the audit.